# 02 — Búsqueda de hiperparámetros: regresión lineal regularizada

La OLS del notebook 01 no regulariza y sufre la colinealidad de las features
mensuales. Acá buscamos la **regularización** óptima sobre una grilla:

- `penalty`: `none` (OLS) · `l2` (Ridge, encoge coeficientes) · `l1` (Lasso, hace
  selección de features) · `elasticnet` (mezcla L1+L2).
- `alpha`: fuerza de la penalización.
- `l1_ratio`: mezcla L1/L2 (solo aplica a elasticnet).

La selección se hace con **validación cruzada temporal** (`evaluacion.buscar`):
folds de ventana expansiva sobre el train, donde la validación es siempre
*posterior* al entrenamiento de cada fold. **El test (≥2021) no se toca en la
búsqueda** — se usa una sola vez, al final, para reportar las métricas del modelo
elegido.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))          # componente_b/ (datos, evaluacion)
warnings.filterwarnings('ignore')                  # silenciar ConvergenceWarning de sklearn

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

import datos, evaluacion as ev
from modelos import LinearRegressor, XGBoostRegressor, NeuralNetRegressor

# Cultivo del estudio (cambiar a 'maiz' para reproducir con maíz).
CULTIVO = 'soja'
ds = datos.prepare(CULTIVO)
print(f'{CULTIVO}: {len(ds.feature_cols)} features | '
      f'train {ds.X_train.shape[0]} filas (≤{datos.TRAIN_END}) | '
      f'test {ds.X_test.shape[0]} filas (≥{datos.TEST_START})')


## La búsqueda

Grid completo, optimizando RMSE de validación.

In [ ]:
grid = {
    'penalty':  ['none', 'ridge', 'lasso', 'elasticnet'],
    'alpha':    [0.1, 1.0, 10.0, 50.0, 100.0],
    'l1_ratio': [0.2, 0.5, 0.8],
}
tabla, best = ev.buscar(LinearRegressor, grid, ds, metric='rmse', n_splits=4)
print('Mejores hiperparámetros:', best)
tabla.head(10)

## Modelo final

Reentrenamos con los mejores hiperparámetros sobre **todo** el train y evaluamos en
el test. Estas son las **métricas del modelo final**.

In [ ]:
modelo = LinearRegressor(**best).fit(ds.X_train, ds.y_train)
pred = modelo.predict(ds.X_test)

m = ev.metricas(ds.y_test, pred)
print('Config final:', modelo.get_config())
print(f"\nMAE  = {m['mae']:.1f} kg/ha")
print(f"RMSE = {m['rmse']:.1f} kg/ha")
print(f"R²   = {m['r2']:.3f}")
print(f"MAPE = {m['mape']:.1f} %")
print(f"\nvs baseline media x depto: RMSE {ev.metricas(ds.y_test, ev.pred_media_depto(ds))['rmse']:.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
ev.plot_pred_vs_real(ds.y_test, pred, f'Lineal {best["penalty"]} (final)',
                     color=ev.C_LINEAR, ax=axes[0])
ev.plot_residuos(ds.y_test, pred, 'Residuos', color=ev.C_LINEAR, ax=axes[1])
plt.tight_layout(); plt.show()

## Coeficientes: qué features sobreviven

Con Lasso/ElasticNet muchos coeficientes van a cero — el modelo se queda con las features informativas.

In [ ]:
coef = getattr(modelo._model, 'coef_', None)
if coef is not None:
    s = pd.Series(coef, index=ds.feature_cols)
    n_cero = int((s.abs() < 1e-8).sum())
    print(f'coeficientes en cero: {n_cero}/{len(s)}')
    print('\nTop 10 |coef|:')
    print(s.reindex(s.abs().sort_values(ascending=False).index).head(10))

## Ambos datasets + el mejor modelo del Componente A

Con la **configuración final ya elegida**, evaluamos en test:

1. dataset `base` (solo clima) vs `era5_ndvi` (clima + NDVI + ERA5), y
2. sobre el que mejor anduvo, tres estrategias que reusan el **mejor detector del
   Componente A** (el VAE `recon_prob`, que aprendió a representar el clima
   "normal"): concatenar su **espacio latente**, hacer la regresión **solo en el
   latente**, y agregar la categórica **`es_anomalo`** (su score umbralado).

El latente/score se computan en `latente.py` (y se cachean). *La primera corrida
entrena el VAE, así que tarda unos minutos.*

In [ ]:
tabla_ds, mejor = ev.comparar_datasets_y_latente(
    LinearRegressor, best, CULTIVO, fixed=None,
    vae_kwargs=dict(score_seeds=(42, 43, 44)))
print('Mejor dataset base:', mejor)
tabla_ds

## Conclusión

La regularización mejora sobre la OLS cruda y da un modelo lineal estable. El techo
del lineal está en su capacidad de capturar solo relaciones **lineales** entre clima
y rinde; los notebooks 03 (XGBoost) y 04 (red neuronal) prueban si lo no-lineal
paga. En cuanto a features: las de ERA5/NDVI y el latente del VAE se evalúan arriba
(ver la conclusión transversal del nb 05).